In [22]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [23]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# My assigned projects
# - 'mpld3_mpld3'
# - 'nilmtk_nilmtk'
# - 'dellytools_delly'
# - 'bedops_bedops'
# - 'opencog_opencog'
# - 'mschauer_causalinference'
# - 'matwey_fips3'
# - 'neuralensemble_libneuroml'
# - 'benjjneb_dada2'
# - 'ngageoint_opensphere'

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['ngageoint_opensphere']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0000b245ec6ef0930251d1efab26a5229976c38b,ngageoint_opensphere


In [24]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|▏         | 20/1486 [00:20<25:27,  1.04s/it]

Got Errors {'0322eef4a5bdd9cd4822eb49e8850e45bb8cb835': 'Key 0322eef4a5bdd9cd4822eb49e8850e45bb8cb835 not found in /da5_fast/All.sha1c/commit_3.tch'}


  9%|▉         | 136/1486 [02:21<23:24,  1.04s/it]

Got Errors {'1620b72459b572a7c3bd562bc1dc748447c28495': 'Key 1620b72459b572a7c3bd562bc1dc748447c28495 not found in /da5_fast/All.sha1c/commit_22.tch'}


 10%|█         | 153/1486 [02:39<23:04,  1.04s/it]

...

Got Errors {'e8f784dcfaa56d4e7bdb1b7de99515948dd501e1': 'Key e8f784dcfaa56d4e7bdb1b7de99515948dd501e1 not found in /da5_fast/All.sha1c/commit_104.tch'}


 99%|█████████▉| 1478/1486 [25:36<00:08,  1.04s/it]

Got Errors {'fe69e1da384c733094e0cb489d73695cf8f4c0c5': 'Key fe69e1da384c733094e0cb489d73695cf8f4c0c5 not found in /da5_fast/All.sha1c/commit_126.tch'}


100%|██████████| 1486/1486 [25:44<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0000b245ec6ef0930251d1efab26a5229976c38b,7ca74da9db702dd28af150248c5b9374a00f8cd4,[679ec891c90963dd56a89bb2245ac77b6a95950a],Glenn Kitchell <kitchellg@wudn-kitchellg.udev....,1607457593,-0700,Glenn Kitchell <kitchellg@wudn-kitchellg.udev....,1607457593,-0700,feat(featureinfo): improve column lookup\n\nup...,0000b245ec6ef0930251d1efab26a5229976c38b,ngageoint_opensphere
1,00052f288aaa70859ca3bdfcdd1f2c7285059233,b00556e87998ea67c6b7427402d5b1261da59d07,[c3159a9d9db64a313e5ea340b7516d5624838ca5],Damien Baca <bacad@wudn-intern4.udev.six3>,1606939645,-0700,Damien Baca <bacad@wudn-intern4.udev.six3>,1607017138,-0700,fix(nobreakspace): normalized whitespace for f...,00052f288aaa70859ca3bdfcdd1f2c7285059233,ngageoint_opensphere


In [25]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0000b245ec6ef0930251d1efab26a5229976c38b', 'tree': '7ca74da9db702dd28af150248c5b9374a00f8cd4', 'parent': ['679ec891c90963dd56a89bb2245ac77b6a95950a'], 'author': 'Glenn Kitchell <kitchellg@wudn-kitchellg.udev.six3>', 'author_time': 1607457593, 'author_tz': '-0700', 'committer': 'Glenn Kitchell <kitchellg@wudn-kitchellg.udev.six3>', 'committer_time': 1607457593, 'committer_tz': '-0700', 'message': 'feat(featureinfo): improve column lookup\n\nupdated so each entry in the featureInfo.simpleProperties.columns only results in one Property being\nreturned.\n'}


In [26]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [27]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,ngageoint_opensphere,0000b245ec6ef0930251d1efab26a5229976c38b,Glenn Kitchell <kitchellg@wudn-kitchellg.udev....,1607457593,feat(featureinfo): improve column lookup\n\nup...


In [28]:
#mode a means append, so you have all your projects in the same file
yournetid='cfinley6'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

# Notes:

While I am uncertain, I think I have the things that were asked for?  Also, only 9 of the 10 repos worked, one of them returned with an error.